# Colorado Tick-Borne Disease Situation Brief
<a id="top"></a>

**AEDES | Advanced Early Disease Prediction and Exploration Service**

This report is designed to answer first: **How common is tick-borne disease right now, and what is my risk?**

Data used here:
- CDC finalized annual Lyme history (currently through 2024)
- CDC provisional in-season weekly indicators (2025/2026 YTD where available)
- iNaturalist research-grade tick observations
- Published Colorado tick phenology and seasonality patterns

**Project Navigation**
- [Home](https://mgifford.github.io/aedesproject-uif/)
- [Notebook Hub](https://mgifford.github.io/aedesproject-uif/notebooks/)
- [Surveillance Dashboard](https://mgifford.github.io/aedesproject-uif/dashboards/)

## Quick Links
- [Current Situation (Today)](#current-situation)
- [Summary of the Analysis](#summary)
- [Lyme Disease Case Trends (Finalized 2015-2024)](#annual-trends)
- [Seasonal Risk Calendar](#seasonal-risk)
- [iNaturalist Tick Observations](#vector-observations)
- [Early Warning Summary](#early-warning)

In [ ]:
from pathlib import Path
import json
import datetime
import calendar
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

TODAY = datetime.date.today().isoformat()

# Resolve project root whether notebook runs from repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "surveillance"


def load_json(filename):
    """Load a surveillance JSON file from common execution locations."""
    candidates = [
        DATA_DIR / filename,
        Path("data/surveillance") / filename,
        Path("../data/surveillance") / filename,
    ]
    for path in candidates:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    return None

print(f'Analysis date: {TODAY}')
print(f'Data directory: {DATA_DIR}')
print(f'Data directory exists: {DATA_DIR.exists()}')

<a id="current-situation"></a>
## Current Situation (Today)

This section prioritizes current-season risk and recency over historical context.

In [ ]:
# Current conditions briefing (today + near-term risk)
year_now = datetime.date.today().year
month_now = datetime.date.today().month
month_name = calendar.month_name[month_now]

# Provisional in-season weekly indicators (current year and prior year)
raw_season_this = load_json(f"{year_now}_season_ytd.json")
raw_season_prev = load_json(f"{year_now - 1}_season_ytd.json")

lyme_ytd_this = None
lyme_ytd_prev = None
wnv_ytd_this = None
if raw_season_this and raw_season_this.get("data"):
    season_this_df = pd.DataFrame(raw_season_this["data"])
    if "lyme_cases" in season_this_df.columns:
        lyme_ytd_this = int(pd.to_numeric(season_this_df["lyme_cases"], errors="coerce").fillna(0).sum())
    if "wnv_cases" in season_this_df.columns:
        wnv_ytd_this = int(pd.to_numeric(season_this_df["wnv_cases"], errors="coerce").fillna(0).sum())

if raw_season_prev and raw_season_prev.get("data"):
    season_prev_df = pd.DataFrame(raw_season_prev["data"])
    if "lyme_cases" in season_prev_df.columns:
        lyme_ytd_prev = int(pd.to_numeric(season_prev_df["lyme_cases"], errors="coerce").fillna(0).sum())

# Most recent iNaturalist signal
raw_ticks_now = load_json('inaturalist_ticks_colorado.json')
inat_tick_obs = len(raw_ticks_now.get("data", [])) if raw_ticks_now else None
inat_fetched = raw_ticks_now.get("fetched", "unknown") if raw_ticks_now else "unknown"

# Practical risk signal for communication
risk_points = 0
if month_now in (4, 5, 6, 7):
    risk_points += 2
elif month_now in (3, 8, 9, 10):
    risk_points += 1

if lyme_ytd_this is not None and lyme_ytd_this > 0:
    risk_points += 1

if inat_tick_obs is not None and inat_tick_obs >= 50:
    risk_points += 1
elif inat_tick_obs is not None and inat_tick_obs >= 10:
    risk_points += 0.5

if risk_points >= 3:
    current_risk = "HIGH"
elif risk_points >= 1.5:
    current_risk = "MODERATE"
else:
    current_risk = "LOW"

yoy_lyme_ytd = None
if lyme_ytd_this is not None and lyme_ytd_prev not in (None, 0):
    yoy_lyme_ytd = round(((lyme_ytd_this - lyme_ytd_prev) / lyme_ytd_prev) * 100, 1)

print("=" * 68)
print(f"CURRENT TICK-BORNE RISK BRIEF ({month_name} {year_now})")
print("=" * 68)
print(f"Current public risk signal: {current_risk}")

if current_risk == "LOW":
    print("Interpretation: Risk appears low right now for most people, though personal exposure still depends on activity and location.")
elif current_risk == "MODERATE":
    print("Interpretation: Conditions support meaningful exposure risk, especially with outdoor activity in brush/grass.")
else:
    print("Interpretation: Conditions are favorable for higher exposure risk; prevention behavior is strongly recommended.")

print("\nMost current indicators:")
print(f"- {year_now} YTD provisional Lyme cases: {lyme_ytd_this if lyme_ytd_this is not None else 'unavailable'}")
print(f"- {year_now} YTD provisional WNV cases (context): {wnv_ytd_this if wnv_ytd_this is not None else 'unavailable'}")
if yoy_lyme_ytd is not None:
    print(f"- Lyme YTD YoY change vs {year_now - 1}: {yoy_lyme_ytd:+.1f}%")
print(f"- iNaturalist research-grade tick observations loaded: {inat_tick_obs if inat_tick_obs is not None else 'unavailable'}")
print(f"- iNaturalist data fetched: {inat_fetched}")

print("\nData recency and source note:")
print("- Finalized annual CDC Lyme tables typically lag and are used for validated historical trend context.")
print("- In-season provisional weekly data is more current for 2025/2026 risk tracking.")
print("- Additional useful sources for Colorado context include CDPHE updates, county public health reports, and veterinary tick surveillance where available.")

<a id="annual-trends"></a>
## 1. Lyme Disease Case Trends (Finalized 2015-2024)

[Back to Home](https://mgifford.github.io/aedesproject-uif/) | [Back to Top](#top)

In [ ]:
raw = load_json('lyme_colorado.json')

if raw and raw.get('data'):
    df_lyme = pd.DataFrame(raw['data'])
    source_label = raw.get('source', 'CDC')
else:
    print('Using built-in historical data')
    source_label = 'CDC Lyme Data Tables (built-in)'
    df_lyme = pd.DataFrame({
        'year':      [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
        'confirmed': [  35,   29,   33,   41,   44,   38,   52,   57,   61,   65],
        'probable':  [  18,   22,   27,   31,   35,   28,   41,   46,   49,   54],
    })

df_lyme['total'] = df_lyme['confirmed'] + df_lyme['probable']
df_lyme['yoy_change'] = (df_lyme['total'].pct_change() * 100).round(1)

print(f'Source: {source_label}')
print(f'Total cases (2015-2024): {df_lyme["total"].sum()}')
print(f'5-year trend (2020-2024): {df_lyme[df_lyme["year"] >= 2020]["total"].sum()} cases')
print(f'Avg annual growth rate: {df_lyme["yoy_change"].dropna().mean():.1f}%')
print('Note: finalized annual CDC Lyme data is currently through 2024; newer signals appear in provisional in-season feeds.')

display_cols = ['year', 'confirmed', 'probable', 'total', 'yoy_change']
df_lyme[display_cols].tail(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Colorado Lyme Disease Surveillance', fontsize=14, fontweight='bold')

# Stacked bar: confirmed vs probable
axes[0].bar(df_lyme['year'], df_lyme['confirmed'], label='Confirmed', color='#2b6cb0', alpha=0.9)
axes[0].bar(df_lyme['year'], df_lyme['probable'], bottom=df_lyme['confirmed'],
            label='Probable', color='#90cdf4', alpha=0.9)
axes[0].set_title('Confirmed + Probable Cases by Year', fontsize=11)
axes[0].set_ylabel('Cases')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xlabel('Year')

# 5-year rolling trend line
axes[1].plot(df_lyme['year'], df_lyme['total'], 'o-', color='#2b6cb0',
             linewidth=2.5, markersize=7, label='Total cases')
if len(df_lyme) >= 3:
    rolling = df_lyme['total'].rolling(3, center=True).mean()
    axes[1].plot(df_lyme['year'], rolling, '--', color='#e53e3e',
                 linewidth=2, label='3-year rolling avg')
axes[1].set_title('Total Cases — Trend', fontsize=11)
axes[1].set_ylabel('Total Cases')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.savefig('lyme_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Source: {source_label}')

<a id="seasonal-risk"></a>
## 2. Seasonal Risk Calendar

[Back to Home](https://mgifford.github.io/aedesproject-uif/) | [Back to Top](#top)

In [ ]:
# Tick activity and disease risk by month for Colorado
# Based on published phenology data from state and CDC surveillance programs
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Risk scores 0-3: 0=none, 1=low, 2=moderate, 3=high
risk_data = {
    'Lyme (I. scapularis)':    [0, 0, 1, 2, 3, 3, 2, 1, 1, 2, 1, 0],
    'RMSF (D. variabilis)':    [0, 0, 1, 2, 3, 2, 1, 1, 1, 1, 0, 0],
    'CTF (D. andersoni)':      [0, 0, 1, 2, 3, 3, 2, 1, 0, 0, 0, 0],
    'Anaplasmosis (Ixodes)':   [0, 0, 1, 2, 3, 3, 2, 1, 1, 2, 1, 0],
    'Tularemia (multi)':       [0, 0, 0, 1, 2, 3, 3, 2, 1, 0, 0, 0],
}

df_risk = pd.DataFrame(risk_data, index=months)

if df_risk.empty or (df_risk.values.sum() == 0):
    print('Seasonal risk calendar skipped: no risk values available for this run.')
else:
    fig, ax = plt.subplots(figsize=(13, 4))
    cmap = plt.cm.RdYlGn_r
    heatmap = ax.imshow(df_risk.T.values, aspect='auto', cmap=cmap, vmin=0, vmax=3,
                        interpolation='nearest')

    ax.set_xticks(range(12))
    ax.set_xticklabels(months, fontsize=10)
    ax.set_yticks(range(len(df_risk.columns)))
    ax.set_yticklabels(df_risk.columns, fontsize=9)
    ax.set_title('Colorado Tick-Borne Disease Seasonal Risk Calendar', fontsize=12, fontweight='bold', pad=12)

    # Annotate cells
    labels = ['-', 'Low', 'Mod', 'High']
    for i in range(len(df_risk.columns)):
        for j in range(12):
            val = int(df_risk.T.values[i, j])
            txt = labels[val]
            ax.text(j, i, txt, ha='center', va='center', fontsize=7.5,
                    color='white' if val == 3 else 'black',
                    fontweight='bold' if val == 3 else 'normal')

    # Mark current month
    current_month_idx = datetime.date.today().month - 1
    ax.axvline(current_month_idx, color='#3182ce', linewidth=2.5, label='Current month')
    ax.legend(loc='upper right', fontsize=9)

    plt.colorbar(heatmap, ax=ax, label='Risk Level', shrink=0.8, ticks=[0, 1, 2, 3])
    plt.tight_layout()
    plt.savefig('tick_seasonal_risk.png', dpi=150, bbox_inches='tight')
    plt.show()

<a id="vector-observations"></a>
## 3. iNaturalist Tick Observations

[Back to Home](https://mgifford.github.io/aedesproject-uif/) | [Back to Top](#top)

In [ ]:
raw_ticks = load_json('inaturalist_ticks_colorado.json')

df_ticks = pd.DataFrame()
if raw_ticks and raw_ticks.get('data') and len(raw_ticks['data']) > 0:
    df_ticks = pd.DataFrame(raw_ticks['data'])
    df_ticks['observed_on'] = pd.to_datetime(df_ticks['observed_on'], errors='coerce')
    df_ticks = df_ticks.dropna(subset=['observed_on'])

if not df_ticks.empty:
    print(f'Total research-grade tick observations in file: {len(df_ticks)}')
    print(f'Fetched: {raw_ticks.get("fetched", "unknown")}')

    if 'taxon' in df_ticks.columns and df_ticks['taxon'].notna().any():
        species_counts = df_ticks['taxon'].value_counts().head(10)

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle('iNaturalist Tick Observations — Colorado', fontsize=13, fontweight='bold')

        # Species breakdown
        colors_sp = plt.cm.tab10(np.linspace(0, 1, len(species_counts)))
        axes[0].barh(species_counts.index[::-1], species_counts.values[::-1], color=colors_sp)
        axes[0].set_title('Observations by Species (Top 10)', fontsize=11)
        axes[0].set_xlabel('Observations')
        axes[0].grid(axis='x', alpha=0.3)

        # Monthly distribution
        df_ticks['month'] = df_ticks['observed_on'].dt.month
        monthly_ticks = df_ticks.groupby('month').size().reindex(range(1, 13), fill_value=0)
        month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
        axes[1].bar(month_names, monthly_ticks.values, color='#744210', alpha=0.8)
        axes[1].set_title('Observations by Month', fontsize=11)
        axes[1].set_ylabel('Observations')
        axes[1].grid(axis='y', alpha=0.3)

        # Highlight current month
        current_month_idx = datetime.date.today().month - 1
        if monthly_ticks.values[current_month_idx] > 0:
            axes[1].patches[current_month_idx].set_edgecolor('#e53e3e')
            axes[1].patches[current_month_idx].set_linewidth(2.5)

        plt.tight_layout()
        plt.savefig('inat_ticks.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('No iNaturalist tick data available (API unavailable or no observations returned).')
    print('Section intentionally left without chart to avoid empty visuals.')

<a id="early-warning"></a>
## 4. Early Warning Summary

[Back to Home](https://mgifford.github.io/aedesproject-uif/) | [Back to Top](#top)

In [ ]:
current_month = datetime.date.today().month
current_month_name = calendar.month_name[current_month]
year_now = datetime.date.today().year

# Overall tick risk this month (max across all diseases)
max_risk = int(df_risk.iloc[current_month - 1].max()) if 'df_risk' in globals() and not df_risk.empty else 0
risk_labels = {0: 'None', 1: 'Low', 2: 'Moderate', 3: 'High'}
risk_label = risk_labels[max_risk]

# Which tick species most active this month
active = []
if 'df_risk' in globals() and not df_risk.empty and current_month_name in df_risk.index:
    active = [disease for disease in df_risk.columns if df_risk.loc[current_month_name, disease] >= 2]

# Current year provisional Lyme and prior-year comparison
raw_season_this = load_json(f"{year_now}_season_ytd.json")
raw_season_prev = load_json(f"{year_now - 1}_season_ytd.json")

lyme_ytd_this = None
lyme_ytd_prev = None
if raw_season_this and raw_season_this.get('data'):
    _df_this = pd.DataFrame(raw_season_this['data'])
    if 'lyme_cases' in _df_this.columns:
        lyme_ytd_this = int(pd.to_numeric(_df_this['lyme_cases'], errors='coerce').fillna(0).sum())
if raw_season_prev and raw_season_prev.get('data'):
    _df_prev = pd.DataFrame(raw_season_prev['data'])
    if 'lyme_cases' in _df_prev.columns:
        lyme_ytd_prev = int(pd.to_numeric(_df_prev['lyme_cases'], errors='coerce').fillna(0).sum())

yoy_hist = float(df_lyme['yoy_change'].iloc[-1]) if 'df_lyme' in globals() and len(df_lyme) > 1 else None
yoy_provisional = None
if lyme_ytd_this is not None and lyme_ytd_prev not in (None, 0):
    yoy_provisional = round(((lyme_ytd_this - lyme_ytd_prev) / lyme_ytd_prev) * 100, 1)

inat_obs = len(df_ticks) if 'df_ticks' in globals() and isinstance(df_ticks, pd.DataFrame) else None

print('=' * 60)
print(f'  AEDES Tick Surveillance Summary — {current_month_name} {year_now}')
print('=' * 60)
print(f'  Current seasonal risk level : {risk_label}')
print()
if active:
    print('  Active disease risks this month:')
    for d in active:
        lvl = risk_labels[int(df_risk.loc[current_month_name, d])]
        print(f'    • {d}: {lvl}')
else:
    print('  No diseases at moderate/high seasonal risk this month')
print()

if lyme_ytd_this is not None:
    print(f'  {year_now} YTD provisional Lyme : {lyme_ytd_this} cases')
if lyme_ytd_prev is not None:
    print(f'  {year_now - 1} YTD provisional Lyme : {lyme_ytd_prev} cases')
if yoy_provisional is not None:
    print(f'  Provisional YTD YoY change  : {yoy_provisional:+.1f}%')

if not df_lyme.empty:
    print(f'  Finalized Lyme (2024)       : {int(df_lyme[df_lyme["year"] == 2024]["total"].sum())} (confirmed + probable)')
    print(f'  Finalized Lyme total        : {int(df_lyme["total"].sum())} cases (2015-2024)')
if yoy_hist is not None:
    print(f'  Finalized YoY (2023→2024)   : {yoy_hist:+.1f}%')

if inat_obs is not None:
    print(f'  iNaturalist tick obs loaded : {inat_obs} (research-grade, CO)')
print()
print('  Key prevention: tick checks after outdoor activity,')
print('  permethrin-treated clothing, avoid tall grass/leaf litter')
print('=' * 60)
print('  Data: CDC NNDSS finalized tables | CDC provisional weekly | iNaturalist | published phenology')
print(f'  Generated: {TODAY}')